# Historical funding-profit explorer

Delta-neutral (long spot + short perp) P&L simulation over the full Postgres history.
All computation is in `bot.analytics.funding_profit`; this notebook is just the viewer.

Sign convention: positive funding_rate -> longs pay shorts -> our book RECEIVES (we are short perp).

In [ ]:
from decimal import Decimal

import polars as pl
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

from bot.analytics.funding_profit import (
    FundingProfitConfig,
    compute_basket,
    list_symbols_with_history,
    simulate_delta_neutral,
)

pio.templates.default = "plotly_dark"
pl.Config.set_tbl_rows(30)

## 1. Single-symbol exploration

In [ ]:
SYMBOL = "BTC/USDT:USDT"

cfg = FundingProfitConfig(symbols=[SYMBOL])
result = simulate_delta_neutral(cfg)[0]

print(f"Symbol: {result.symbol}")
print(f"Has spot data: {result.has_spot_data}")
print(f"Warnings: {result.warnings}")
print(f"Funding events: {result.summary['funding_events']}")
print(f"Coverage: {result.summary['start']} -> {result.summary['end']}")

In [ ]:
def render_symbol_dashboard(result, title_prefix=""):
    """2x2 grid: cumulative funding, equity curve, distribution, drawdown."""
    eq = result.equity_curve
    fdf = result.funding_df

    fig = make_subplots(
        rows=2,
        cols=2,
        subplot_titles=(
            "Cumulative funding received (USD)",
            "Equity curve: gross funding vs. net of fees",
            "Per-funding-event payment distribution (USD)",
            "Drawdown % from running peak",
        ),
        vertical_spacing=0.14,
        horizontal_spacing=0.09,
    )

    # 1. Cumulative funding
    fig.add_trace(
        go.Scatter(
            x=fdf["funding_time"].to_list(),
            y=fdf["cumulative_funding"].to_list(),
            name="Cumulative funding",
            mode="lines",
            line=dict(color="#00cc96", width=2),
            hovertemplate="%{x|%Y-%m-%d %H:%M}<br>Cumulative: $%{y:,.2f}<extra></extra>",
        ),
        row=1, col=1,
    )

    # 2. Equity curve: gross (initial + funding_pnl) vs net
    initial = float(result.config.initial_capital_usd)
    gross_series = (eq["funding_pnl"] + initial).to_list()
    fig.add_trace(
        go.Scatter(
            x=eq["time"].to_list(),
            y=gross_series,
            name="Gross (funding only)",
            mode="lines",
            line=dict(color="#636efa", width=2),
            hovertemplate="%{x|%Y-%m-%d %H:%M}<br>Gross: $%{y:,.2f}<extra></extra>",
        ),
        row=1, col=2,
    )
    fig.add_trace(
        go.Scatter(
            x=eq["time"].to_list(),
            y=eq["equity"].to_list(),
            name="Net (funding - fees - basis)",
            mode="lines",
            line=dict(color="#ef553b", width=2),
            hovertemplate="%{x|%Y-%m-%d %H:%M}<br>Net: $%{y:,.2f}<extra></extra>",
        ),
        row=1, col=2,
    )

    # 3. Per-funding-event payment distribution.
    mean_v = float(fdf["funding_payment"].mean() or 0.0)
    median_v = float(fdf["funding_payment"].median() or 0.0)
    std_v = float(fdf["funding_payment"].std() or 0.0)
    fig.add_trace(
        go.Histogram(
            x=fdf["funding_payment"].to_list(),
            nbinsx=80,
            name="Per-event payment",
            marker=dict(color="#ab63fa"),
            hovertemplate="Bin: $%{x:,.2f}<br>Count: %{y}<extra></extra>",
        ),
        row=2, col=1,
    )
    fig.add_vline(
        x=mean_v,
        line=dict(color="#00cc96", dash="dash"),
        row=2, col=1,
        annotation=dict(text=f"mean ${mean_v:,.2f}", font=dict(color="#00cc96")),
    )
    fig.add_vline(
        x=median_v,
        line=dict(color="#ffa15a", dash="dot"),
        row=2, col=1,
        annotation=dict(text=f"median ${median_v:,.2f}", font=dict(color="#ffa15a")),
    )

    # 4. Drawdown.
    dd_pct_series = (eq["drawdown"] * 100.0).to_list()
    fig.add_trace(
        go.Scatter(
            x=eq["time"].to_list(),
            y=dd_pct_series,
            name="Drawdown %",
            mode="lines",
            fill="tozeroy",
            line=dict(color="#ef553b", width=1),
            fillcolor="rgba(239,85,59,0.3)",
            hovertemplate="%{x|%Y-%m-%d %H:%M}<br>DD: %{y:.3f}%%<extra></extra>",
        ),
        row=2, col=2,
    )

    # Shade the worst drawdown run.
    eq_list = eq["equity"].to_list()
    times = eq["time"].to_list()
    running_max_val = -1e18
    worst_start = worst_end = None
    worst_dd = 0.0
    cur_start = None
    cur_peak = eq_list[0] if eq_list else 0
    for i, v in enumerate(eq_list):
        if v >= cur_peak:
            cur_peak = v
            cur_start = None
            continue
        if cur_start is None:
            cur_start = i
        dd_here = (v - cur_peak) / cur_peak
        if dd_here < worst_dd:
            worst_dd = dd_here
            worst_start = cur_start
            worst_end = i
    if worst_start is not None and worst_end is not None:
        fig.add_vrect(
            x0=times[worst_start], x1=times[worst_end],
            fillcolor="rgba(255, 200, 0, 0.12)",
            line_width=0,
            row=2, col=2,
            annotation=dict(
                text=f"worst DD {worst_dd*100:.2f}%",
                font=dict(color="#ffd166"),
            ),
        )

    fig.update_xaxes(title_text="Date", row=1, col=1)
    fig.update_yaxes(title_text="USD", row=1, col=1)
    fig.update_xaxes(title_text="Date", row=1, col=2)
    fig.update_yaxes(title_text="Equity (USD)", row=1, col=2)
    fig.update_xaxes(title_text="Payment (USD)", row=2, col=1)
    fig.update_yaxes(title_text="Count", row=2, col=1)
    fig.update_xaxes(title_text="Date", row=2, col=2)
    fig.update_yaxes(title_text="Drawdown %", row=2, col=2)

    fig.update_layout(
        title=f"{title_prefix}{result.symbol} -- delta-neutral P&L (std={std_v:,.2f})",
        template="plotly_dark",
        height=820,
        showlegend=True,
        legend=dict(orientation="h", y=-0.08),
    )
    return fig

fig_single = render_symbol_dashboard(result)
fig_single.show()

## 2. Summary table

In [ ]:
def summary_row(s):
    return {
        "symbol": s["symbol"],
        "events": s["funding_events"],
        "total_funding_usd": round(s["total_funding"], 2),
        "total_fees_usd": round(s["total_fees"], 2),
        "price_pnl_usd": round(s["price_pnl"], 2),
        "net_pnl_usd": round(s["net_pnl"], 2),
        "max_dd_pct": round(s["max_drawdown_pct"], 3),
        "max_dd_days": round(s["max_drawdown_duration_days"], 1),
        "mean_rate": round(s["mean_funding_rate"], 6),
        "median_rate": round(s["median_funding_rate"], 6),
        "pct_pos": round(s["pct_positive_events"], 2),
        "monotonic": s["is_monotonically_profitable"],
        "sharpe_like": round(s["sharpe_like"], 2),
        "has_spot": s["has_spot_data"],
    }

summary_df = pl.DataFrame([summary_row(result.summary)])
print(summary_df)

## 3. Multi-symbol basket

In [ ]:
BASKET = ["BTC/USDT:USDT", "ETH/USDT:USDT", "SOL/USDT:USDT"]

basket_cfg = FundingProfitConfig(symbols=BASKET)
basket_results = simulate_delta_neutral(basket_cfg)
basket = compute_basket(basket_results)  # equal-weighted

print("Per-symbol summary:")
per_sym = pl.DataFrame([summary_row(r.summary) for r in basket_results])
print(per_sym)
print()
print("Basket summary:")
print(pl.DataFrame([summary_row(basket.summary)]))

In [ ]:
# Rebuild basket funding_df shape so render_symbol_dashboard works off the
# basket (no cumulative_funding column yet -- we re-derive from equity curve).
basket_fdf = basket.equity_curve.select(
    pl.col("time").alias("funding_time"),
    pl.col("funding_pnl").alias("cumulative_funding"),
)
# For the histogram we want per-event payments from the concatenated stream.
if "funding_payment_weighted" in basket.funding_df.columns:
    basket_events = basket.funding_df.select(
        pl.col("funding_time"),
        pl.col("funding_payment_weighted").alias("funding_payment"),
        pl.col("funding_payment_weighted").cum_sum().alias("cumulative_funding"),
    )
else:
    basket_events = basket_fdf.with_columns(pl.col("cumulative_funding").diff().fill_null(0.0).alias("funding_payment"))

# Patch a lightweight result for rendering.
import copy
basket_render = copy.copy(basket)
basket_render.funding_df = basket_events

fig_basket = render_symbol_dashboard(basket_render, title_prefix="BASKET -- ")
fig_basket.show()

In [ ]:
# Per-symbol equity curves overlaid.
fig_overlay = go.Figure()
for r in basket_results:
    if r.equity_curve.is_empty():
        continue
    fig_overlay.add_trace(
        go.Scatter(
            x=r.equity_curve["time"].to_list(),
            y=r.equity_curve["equity"].to_list(),
            name=r.symbol,
            mode="lines",
            hovertemplate="%{x|%Y-%m-%d %H:%M}<br>Equity: $%{y:,.2f}<extra>" + r.symbol + "</extra>",
        )
    )
fig_overlay.update_layout(
    title="Per-symbol equity curves",
    template="plotly_dark",
    xaxis_title="Date",
    yaxis_title="Equity (USD)",
    height=420,
)
fig_overlay.show()

## 4. Top-N / bottom-N ranking

All symbols with >= 1 year of history. Ranked by Sharpe-like = `net_pnl / std(payment) * sqrt(events/year)`.

In [ ]:
elig = list_symbols_with_history(min_days=365)
print(f"Eligible symbols: {elig.height}")

symbols = elig["symbol"].to_list()
all_cfg = FundingProfitConfig(symbols=symbols)
all_results = simulate_delta_neutral(all_cfg)
print(f"Simulated: {len(all_results)} symbols")

In [ ]:
ranking_rows = []
for r in all_results:
    s = r.summary
    if s["funding_events"] == 0:
        continue
    ranking_rows.append(summary_row(s))

ranking = pl.DataFrame(ranking_rows).sort("sharpe_like", descending=True)
print("Top 20 by Sharpe-like:")
print(ranking.head(20))
print()
print("Bottom 10 by Sharpe-like:")
print(ranking.tail(10))